In [ ]:
import os
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
from matplotlib import colormaps
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
import seaborn as sns
sns.set_theme()

In [ ]:
filepath = os.getcwd()

# Variables to be changed with each run
filename = f"{filepath}\\Ridgecrest64_Waze_withMMI.csv"
city = "Ridgecrest Mw 6.4" # To be displayed on figures
eq_weekday = "Thursday"
eq_weekday_num = 3 # Monday = 0, Sunday = 6
data_source = 'Waze' # 'Waze' or 'CHP'

In [ ]:
# Read in csv file of earthquake data, including earthquake time, accident time, and MMI
if data_source == 'Waze':
    earthquake = pd.read_csv(filename,
                             usecols = ['pub_utc_timestamp', 'mmi_value', 'eq_time', 'alert_type'],
                             parse_dates = ['pub_utc_timestamp', 'eq_time'], on_bad_lines = 'warn', engine = 'python')
    earthquake = earthquake[(earthquake['alert_type'] == 'ACCIDENT')]
    earthquake['pub_utc_timestamp'] = pd.to_datetime(earthquake['pub_utc_timestamp'], utc = True)
    earthquake = earthquake.rename(columns = {'pub_utc_timestamp': 'accident_time_gmt'})
else:
    earthquake = pd.read_csv(filename,
                             usecols = ['accident_time_gmt', 'mmi_value', 'eq_time'],
                             parse_dates = ['accident_time_gmt', 'eq_time'])
    earthquake['accident_time_gmt'] = pd.to_datetime(earthquake['accident_time_gmt'], utc = True)

# Define datetimes
earthquake['eq_time'] = pd.to_datetime(earthquake.eq_time, utc = True) # origin time UTC
earthquake = earthquake.sort_values(by = ['accident_time_gmt'])
earthquake["seconds_of_week"] = [math.fmod(ds, 60*60*168) for ds in (earthquake['accident_time_gmt'] - earthquake['eq_time']).dt.total_seconds()]
earthquake.loc[earthquake['seconds_of_week'] >= 0, 'seconds_of_week'] = None
earthquake.loc[earthquake['seconds_of_week'] < -302400, 'seconds_of_week'] = earthquake.loc[earthquake['seconds_of_week'] < -302400, 'seconds_of_week'] + 604800
earthquake["week_of_year"] = [(int(ds)//int(60*60*168)) +1 for ds in (earthquake['accident_time_gmt'] - earthquake['eq_time']).dt.total_seconds()]

# Define static variables
origintime = pd.to_datetime(earthquake.eq_time.unique()[0], utc = True) # origin time UTC
if filename == "NJNYC_Waze_withMMI.csv":
    origintime_local = origintime.astimezone(ZoneInfo('America/New_York'))
else:
    origintime_local = origintime.astimezone(ZoneInfo('America/Los_Angeles')) # origin time in the local time zone
mmi_bins = list(np.arange(2.0, 7.25, 0.25)) # minimum MMI of 2.0
num_MMIs = len(mmi_bins)
T0s = [-3.0, -2.0, -1.0, 0.0, 1.0, 2.0] # break time between before/after in hours relative to origin time
num_T0s = len(T0s)
window = 2.0  # windows in hours

Ratio = np.zeros((num_MMIs, num_T0s)) # ratio of crashes after T0 to before T0, for each T0 and MMI bin

In [ ]:
# 1: plot accident counts by MMI over time
# NOTE: for the times where no physical shaking was felt (i.e., times other than immediately after an earthquake), "MMI" refers to the MMI that was felt at the same location at the time of the earthquake. It represents a location, rather than the literal MMI at the time specified.
fig, ax = plt.subplots(figsize = (10, 8))
# Determine after and before windows relative to T0
for x in range(num_T0s):
    dt0=T0s[x]
    bin_center = origintime + timedelta(hours = dt0)
    OTP2 = bin_center + timedelta(hours = window) # bin center plus 2 hours (window)
    OTM2 = bin_center - timedelta(hours = window) # bin center minus 2 hours (window)
    # Find crashes above the minimum MMI that fall within the after and before windows
    for y in range(num_MMIs):
        mmi_lim = mmi_bins[y]
        KK = earthquake[(earthquake['mmi_value'] >= mmi_lim) & (earthquake['mmi_value'] < 100)] # limting the MMI to less than 100 removes erroneous outliers
        KKP2 = KK[(KK["accident_time_gmt"] >= bin_center) & (KK["accident_time_gmt"] < OTP2) & (KK["mmi_value"] >= mmi_lim)]
        KKM2 = KK[(KK["accident_time_gmt"] >= OTM2) & (KK["accident_time_gmt"] < bin_center) & (KK["mmi_value"] >= mmi_lim)]
        # populate Ratio array
        eqs_KKP2 = len(KKP2)
        eqs_KKM2 = len(KKM2)
        if (eqs_KKP2 < 2):
            Ratio[y, x] = np.nan
        else:
            Ratio[y, x] = eqs_KKP2
        ax.scatter(KK['accident_time_gmt'], KK['mmi_value'])
ax.set(xlabel = 'Date UTC',
       ylabel = 'Accident MMI')
plt.show()

In [ ]:
# 2: plot cumulative total accident counts
fig, ax = plt.subplots(figsize = (10, 8))
times = earthquake['accident_time_gmt'].tolist()
counts = np.arange(0, len(times))
ax.plot(times, counts)
plt.vlines(origintime, 0, max(counts), color = 'r', linestyle = '--')
ax.set(xlabel = 'Date UTC',
       ylabel = 'Cumulative Accidents')
plt.legend(["Accident Count", f"{city} Earthquake"], loc = "best")
plt.show()

In [ ]:
# 3: plot moving window ratio with normalization
fig, ax = plt.subplots(figsize = (10, 8))
for x in range(num_T0s):
    plt.plot(mmi_bins, np.divide(Ratio[:, x], Ratio[0,x]), 'o-', linewidth = 2, label = T0s[x])
# colors
colormap = plt.cm.viridis   
colors = [colormap(i) for i in np.linspace(0, 1, len(ax.lines))]
for i, j in enumerate(ax.lines):
    j.set_color(colors[i])
ax.set(ylabel = 'Ratio of Crashes After to Before (Normalized at 2 MMI)',
       xlabel = 'MMI values',
       xlim = [1.9, 7.1])
plt.legend(loc = "best")
plt.show()

In [ ]:
# 4a: plot moving window accident counts WITHOUT normalization
fig = plt.figure(figsize = (10, 8))
fig.suptitle(f'{city} Earthquake to Control Crash Ratios - {data_source}')
ax1 = plt.subplot(222)
ax1.set_title('Earthquake Accident Counts')
for x in range(num_T0s):
    plt.plot(mmi_bins, Ratio[:, x], 'o-', linewidth = 2, label = T0s[x])
# colors
colormap = plt.cm.viridis   
colors = [colormap(i) for i in np.linspace(0, 1, len(ax1.lines))]
for i, j in enumerate(ax1.lines):
    j.set_color(colors[i])
ax1.set(xlabel = 'Lower MMI Limit',
       xlim = [1.9, 7.1])
ax1.tick_params(left = False, labelleft = False) 

# 4b: plot control accident counts for every week in the year prior
Ratio_control = np.zeros((num_MMIs, num_T0s))
for x in range(num_T0s):
    dt0 = T0s[x]
    bin_center = dt0 * 3600
    OTP2 = bin_center + window * 3600 # bin center plus 2 hours (window)
    OTM2 = bin_center - window * 3600 # bin center minus 2 hours (window)
    # find crashes above the minimum MMI that fall within the windows
    for y in range(num_MMIs):
        mmi_lim = mmi_bins[y]
        KK = earthquake[(earthquake['mmi_value'] >= mmi_lim) & (earthquake['mmi_value'] < 100)] # limting the MMI to less than 100 removes erroneous outliers
        KKP2 = KK[(KK["seconds_of_week"] >= bin_center) & (KK["seconds_of_week"] < OTP2) & (KK["mmi_value"] >= mmi_lim)]
        KKM2 = KK[(KK["seconds_of_week"] >= OTM2) & (KK["seconds_of_week"] < bin_center) & (KK["mmi_value"] >= mmi_lim)]
        KKP2[KKP2['week_of_year']<0].groupby('week_of_year', as_index=False)['mmi_value'].count()['mmi_value'].mean()
        # populate Ratio array
        eqs_KKP2 = KKP2[KKP2['week_of_year']<0].groupby('week_of_year', as_index=False)['mmi_value'].count()['mmi_value'].mean()
        eqs_KKM2 = len(KKM2)
        if (eqs_KKP2 < 1):
            Ratio_control[y, x] = np.nan
        else:
            Ratio_control[y, x] = eqs_KKP2       
ax2 = plt.subplot(221, sharey = ax1)
ax2.set_title('Control Accident Counts')
for x in range(num_T0s):
    plt.plot(mmi_bins, Ratio_control[:, x], 'o-', linewidth = 2, label = T0s[x])
# colors
colormap = plt.cm.viridis   
colors = [colormap(i) for i in np.linspace(0, 1, len(ax2.lines))]
for i, j in enumerate(ax2.lines):
    j.set_color(colors[i])
ax2.set(ylabel = 'Number of Crashes',
       xlabel = 'Lower MMI Limit',
       xlim = [1.9, 7.1])

# 4c: plot moving window ratio with normalization to the control period
ax3 = plt.subplot(212)
ax3.set_title('Ratios of Crashes')
for x in range(num_T0s):
    After_Control_Ratio = np.divide(Ratio[:, x], Ratio_control[:,x])
    plt.plot(mmi_bins, After_Control_Ratio, 'o-', linewidth = 2, label = f'{T0s[x]} hours ({(origintime_local + timedelta(hours = T0s[x])).strftime("%H:%M")}) to {T0s[x]+window} hours ({(origintime_local + timedelta(hours = T0s[x] + window)).strftime("%H:%M")})')
# colors
colormap = plt.cm.viridis   
colors = [colormap(i) for i in np.linspace(0, 1, len(ax3.lines))]
for i, j in enumerate(ax3.lines):
    j.set_color(colors[i])
ax3.set(ylabel = 'Crash Ratio',
       xlabel = 'Lower MMI Limit',
       xlim = [1.9, 7.1])
# set subplot and legend sizes and locations
plt.subplots_adjust(hspace = 0.3, wspace = 0.2)
box1 = ax1.get_position()
ax1.set_position([box1.x0, box1.y0 + box1.height * 0.3,
                 box1.width, box1.height * 0.8])
box2 = ax2.get_position()
ax2.set_position([box2.x0, box2.y0 + box2.height * 0.3,
                 box2.width, box2.height * 0.8])
box3 = ax3.get_position()
ax3.set_position([box3.x0, box3.y0 + box3.height * 0.5,
                 box3.width, box3.height * 0.8])
plt.legend(loc = 'upper center', bbox_to_anchor = (0.5, -0.3),
          fancybox = False, shadow = False, ncol = 3)
plt.show()